# Budgerigar：内容 Token Memory 预训练

层级声学模型仍坍缩到平均模板。本阶段暂时移除声学复读目标，只要求多速率 token bank 从音频恢复 transcript，并通过 InfoNCE 匹配正确文字。只有内容记忆通过后才重新接入声学解码器。

In [ ]:
#@title 1. 更新项目与安装依赖
REPO_DIR='/content/Budgerigar'
from pathlib import Path
import subprocess,sys,importlib
if not Path(REPO_DIR).is_dir(): subprocess.run(['git','clone','--depth=1','https://github.com/DoctorAwe/Budgerigar.git',REPO_DIR],check=True)
else: subprocess.run(['git','-C',REPO_DIR,'pull','--ff-only'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-e',f'{REPO_DIR}[train,data]'],check=True)
sys.path.insert(0,REPO_DIR)
for name in [key for key in list(sys.modules) if key=='budgerigar' or key.startswith('budgerigar.')]: del sys.modules[name]
importlib.invalidate_caches()
commit=subprocess.run(['git','-C',REPO_DIR,'rev-parse','--short','HEAD'],capture_output=True,text=True,check=True).stdout.strip()
print('commit:',commit)

In [ ]:
#@title 2. 挂载 Drive 并复用特征统计
from google.colab import drive
drive.mount('/content/drive')
WORK_ROOT=Path('/content/drive/MyDrive/Budgerigar')
FEATURE_FINGERPRINT='f1f2ace085a17835' #@param {type:'string'}
TARGET_SPEAKER='arctic_slt' # 仅用于找到已有统计缓存，本阶段内容数据不依赖目标声线
FEATURE_MANIFEST=WORK_ROOT/'manifests'/f'cmu_arctic.features.{FEATURE_FINGERPRINT}.jsonl'
STATS_PATH=WORK_ROOT/'features'/f'stats.smoke64.{TARGET_SPEAKER}.{FEATURE_FINGERPRINT}.pt'
assert FEATURE_MANIFEST.is_file(),FEATURE_MANIFEST
assert STATS_PATH.is_file(),STATS_PATH
import torch,json
stats=torch.load(STATS_PATH,map_location='cpu',weights_only=True)

In [ ]:
#@title 3. CTC 数据容量检查
from budgerigar.content_data import CharacterVocabulary,ContentFeatureDataset
vocab=CharacterVocabulary()
train_preview=ContentFeatureDataset(FEATURE_MANIFEST,'train',stats,vocab,max_records=8,preload=False,update_stride=4,token_slots=128)
validation_preview=ContentFeatureDataset(FEATURE_MANIFEST,'validation',stats,vocab,max_records=8,preload=False,update_stride=4,token_slots=128)
print('vocabulary:',len(vocab.symbols),vocab.symbols)
print('preview train/validation:',len(train_preview),len(validation_preview))
sample=train_preview[0]
print(sample[2],sample[0].shape,len(sample[1]),sample[3])

In [ ]:
#@title 4. T4 内容记忆 smoke training
MAX_STEPS=300 #@param {type:'integer'}
BATCH_SIZE=4 #@param {type:'integer'}
TOKEN_SLOTS=128 #@param {type:'integer'}
UPDATE_STRIDE=4 #@param {type:'integer'}
if not torch.cuda.is_available(): raise RuntimeError('请选择 GPU runtime')
from budgerigar.content_memory import ContentMemoryConfig
from budgerigar.train_content_memory import ContentTrainingConfig,train_content_memory
RUN_DIR=WORK_ROOT/'checkpoints'/f'content_memory_{FEATURE_FINGERPRINT}'
model_config=ContentMemoryConfig(token_slots=TOKEN_SLOTS,update_stride=UPDATE_STRIDE,vocabulary_size=len(vocab.symbols))
training=ContentTrainingConfig(batch_size=BATCH_SIZE,max_steps=MAX_STEPS)
report=train_content_memory(FEATURE_MANIFEST,stats,RUN_DIR,training,model_config)
print(json.dumps(report,ensure_ascii=False,indent=2))

In [ ]:
#@title 5. 内容门槛初判
best=min(report['history'],key=lambda row:row['validation_cer'])
print(json.dumps(best,ensure_ascii=False,indent=2))
print('初步门槛：CER < 0.8 且 batch retrieval Top-1 > 0.5')
print('smoke_pass =',best['validation_cer']<0.8 and best['validation_retrieval_top1']>0.5)

In [ ]:
#@title 6. 保存运行元数据
from budgerigar.experiment import write_run_metadata
metadata=write_run_metadata(RUN_DIR/'run_metadata.json',FEATURE_MANIFEST,{'architecture':'content_token_memory','token_slots':TOKEN_SLOTS,'update_stride':UPDATE_STRIDE,'best_validation_cer':report['best_validation_cer']},repository=REPO_DIR)
print(metadata.read_text(encoding='utf-8'))